# Montagem da base de conhecimento

Este notebook mostra, passo a passo, como as tabelas originais da Olist viram a base usada no projeto. Ao final temos dois arquivos em `data/processed/`:

- `reviews.parquet`: todas as avaliações, uma por linha, já com as informações do pedido;
- `documents.parquet`: só as avaliações com comentário útil, que são as usadas nas buscas.

Na prática esses arquivos são gerados pelo script `src/data/build_documents.py` (comando `make data`). Aqui refazemos as mesmas etapas de forma explícita, olhando o resultado de cada uma, e no fim conferimos que chegamos ao mesmo resultado do script.

As análises que justificam várias das escolhas (por exemplo, o tamanho dos comentários e a quantidade de avaliações sem texto) estão no notebook `02_eda_reviews.ipynb`.

## 1. Preparando o ambiente

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data.load_olist import load_olist
from src.data.clean import juntar_titulo_mensagem, normalizar, informativo, contar_palavras
from src.data.pii import mascarar_pii

pd.set_option("display.max_colwidth", 80)
t = load_olist(ROOT / "data" / "raw")
for nome, df in t.items():
    print(f"{nome:12s} {df.shape[0]:>7,} linhas x {df.shape[1]:>2} colunas")

reviews       99,224 linhas x  7 colunas
orders        99,441 linhas x  8 colunas
items        112,650 linhas x  7 colunas
products      32,951 linhas x  9 colunas
categorias        71 linhas x  2 colunas
customers     99,441 linhas x  5 colunas


## 2. Como as tabelas se ligam

O dataset segue um modelo relacional: cada tabela guarda um assunto, e elas se conectam por códigos. O diagrama do dataset no Kaggle mostra essas ligações. Para o nosso caso, o caminho é este:

```
avaliações --order_id--> pedidos --customer_id--> clientes
                            |
                         order_id
                            v
                    itens do pedido --product_id--> produtos --categoria--> tradução
```

Um ponto importante: a avaliação está ligada ao **pedido**, e não ao produto. Para saber a categoria do produto avaliado precisamos passar pelos itens do pedido, e um pedido pode ter vários itens.

In [2]:
t["reviews"][["review_id", "order_id", "review_score", "review_comment_message"]].head(3)

,review_id,order_id,review_score,review_comment_message
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN


In [3]:
t["items"][["order_id", "order_item_id", "product_id", "price", "freight_value"]].head(3)

,order_id,order_item_id,product_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.0,17.87


## 3. O problema de juntar tudo direto

A forma mais simples seria juntar as avaliações com os itens pelo `order_id`. O problema aparece em pedidos com mais de um item. Veja este pedido real, que tem três itens de duas categorias:

In [4]:
pedido = "32638fbca8d1a197e649fd5138e12049"
itens = t["items"].merge(t["products"][["product_id", "product_category_name"]], on="product_id", how="left")
itens[itens.order_id == pedido][["order_id", "order_item_id", "product_category_name", "price", "freight_value"]]

,order_id,order_item_id,product_category_name,price,freight_value
22055,32638fbca8d1a197e649fd5138e12049,1,ferramentas_jardim,22.9,8.40
22056,32638fbca8d1a197e649fd5138e12049,2,moveis_decoracao,93.5,59.28
22057,32638fbca8d1a197e649fd5138e12049,3,moveis_decoracao,93.5,59.28


In [5]:
junta_direto = t["reviews"].merge(itens, on="order_id")
junta_direto[junta_direto.order_id == pedido][["review_id", "review_score", "product_category_name", "price"]]

,review_id,review_score,product_category_name,price
77,6a0969d0769d2f3ff6438809a1cc455e,5,ferramentas_jardim,22.9
78,6a0969d0769d2f3ff6438809a1cc455e,5,moveis_decoracao,93.5
79,6a0969d0769d2f3ff6438809a1cc455e,5,moveis_decoracao,93.5


A mesma avaliação apareceu **três vezes**, uma para cada item. Se fizéssemos isso para a base inteira, os comentários de pedidos com vários itens seriam contados várias vezes nas buscas e nas estatísticas.

A solução é **resumir os itens por pedido antes de juntar**: cada pedido vira uma única linha, e só então essa linha é ligada à avaliação.

## 4. Passo 1: resumir os itens de cada pedido

Para cada pedido calculamos:

- `valor_total`: soma de preço e frete de todos os itens;
- `n_itens`: quantidade de itens;
- `categoria_pt` e `categoria_en`: a categoria do item de **maior valor**, que tratamos como o produto principal do pedido;
- `multi_categoria`: indica se o pedido tem produtos de mais de uma categoria.

Usamos a categoria do item mais caro porque precisamos de um único valor por pedido para poder filtrar. A coluna `multi_categoria` deixa registrado quando essa simplificação acontece. Produtos sem categoria recebem `sem_categoria`.

In [6]:
itens = itens.merge(t["categorias"], on="product_category_name", how="left")
itens["product_category_name"] = itens["product_category_name"].fillna("sem_categoria")
itens["product_category_name_english"] = itens["product_category_name_english"].fillna("no_category")
itens["valor_item"] = itens["price"] + itens["freight_value"]

principal = (
    itens.sort_values("valor_item", ascending=False)
    .drop_duplicates("order_id")
    [["order_id", "product_category_name", "product_category_name_english"]]
    .rename(columns={"product_category_name": "categoria_pt", "product_category_name_english": "categoria_en"})
)
resumo = itens.groupby("order_id").agg(
    valor_total=("valor_item", "sum"),
    n_itens=("order_item_id", "count"),
    n_categorias=("product_category_name", "nunique"),
).reset_index()
resumo_itens = principal.merge(resumo, on="order_id")
resumo_itens["multi_categoria"] = resumo_itens["n_categorias"] > 1
resumo_itens[resumo_itens.order_id == pedido]

,order_id,categoria_pt,categoria_en,valor_total,n_itens,n_categorias,multi_categoria
27644,32638fbca8d1a197e649fd5138e12049,moveis_decoracao,furniture_decor,336.86,3,2,True


Agora o pedido do exemplo é uma linha só: valor total de R$ 336,86, três itens, categoria principal `moveis_decoracao` (os itens mais caros) e a marcação de que há mais de uma categoria.

## 5. Passo 2: informações do pedido e da entrega

Juntamos o resumo dos itens com a tabela de pedidos e com o estado do cliente, e calculamos as informações de entrega:

- `entregue`: se existe data de entrega ao cliente;
- `dias_atraso`: diferença em dias entre a entrega real e a prevista (positivo significa atraso);
- `entregue_com_atraso`: se `dias_atraso` é maior que zero;
- `ano_mes_compra`: mês da compra no formato aaaamm, usado nos filtros de período.

Pedidos cancelados ou ainda não entregues ficam com `entregue = False` e `dias_atraso` vazio. Preferimos deixar vazio a inventar uma data.

In [7]:
pedidos = t["orders"].merge(t["customers"][["customer_id", "customer_state"]], on="customer_id", how="left")
pedidos = pedidos.merge(resumo_itens, on="order_id", how="left")
pedidos["categoria_pt"] = pedidos["categoria_pt"].fillna("sem_categoria")
pedidos["categoria_en"] = pedidos["categoria_en"].fillna("no_category")
pedidos["n_itens"] = pedidos["n_itens"].fillna(0).astype(int)
pedidos["multi_categoria"] = pedidos["multi_categoria"].fillna(False).astype(bool)

entrega = pedidos["order_delivered_customer_date"]
prevista = pedidos["order_estimated_delivery_date"]
pedidos["entregue"] = entrega.notna()
atraso = (entrega.dt.normalize() - prevista.dt.normalize()).dt.days
pedidos["dias_atraso"] = atraso.astype("Int64")
pedidos["entregue_com_atraso"] = (atraso > 0).fillna(False)
pedidos["ano_mes_compra"] = pd.to_numeric(pedidos["order_purchase_timestamp"].dt.strftime("%Y%m"), errors="coerce").astype("Int64")
pedidos = pedidos.rename(columns={"customer_state": "uf_cliente"})

pedidos[["order_id", "order_status", "order_estimated_delivery_date", "order_delivered_customer_date",
         "entregue", "dias_atraso", "entregue_com_atraso", "ano_mes_compra"]].head()

,order_id,order_status,order_estimated_delivery_date,order_delivered_customer_date,entregue,dias_atraso,entregue_com_atraso,ano_mes_compra
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-18,2017-10-10 21:25:13,True,-8,False,201710
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-13,2018-08-07 15:27:45,True,-6,False,201807
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-09-04,2018-08-17 18:06:29,True,-18,False,201808
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-12-15,2017-12-02 00:28:42,True,-13,False,201711
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-26,2018-02-16 18:17:02,True,-10,False,201802


In [8]:
pedidos[~pedidos.entregue][["order_id", "order_status", "entregue", "dias_atraso"]].head(3)

,order_id,order_status,entregue,dias_atraso
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,False,<NA>
44,ee64d42b8cf066f35eac1cf57de1aa85,shipped,False,<NA>
103,0760a852e4e9d89eb77bf631eaaf1c84,invoiced,False,<NA>


## 6. Passo 3: ligar as avaliações aos pedidos

Agora juntamos as avaliações com a tabela de pedidos. Aqui aparece outro caso: algumas avaliações (mesmo `review_id` e mesmo texto) estão ligadas a mais de um pedido.

In [9]:
avaliacoes = t["reviews"].merge(pedidos, on="order_id", how="left")
repetidas = avaliacoes.groupby("review_id").order_id.nunique()
print("avaliações ligadas a mais de um pedido:", (repetidas > 1).sum())

exemplo = repetidas[repetidas > 1].index[0]
avaliacoes[avaliacoes.review_id == exemplo][["review_id", "order_id", "order_purchase_timestamp", "categoria_pt", "review_comment_message"]]

avaliações ligadas a mais de um pedido: 789


,review_id,order_id,order_purchase_timestamp,categoria_pt,review_comment_message
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,2018-02-02 18:01:10,informatica_acessorios,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequente..."
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,2018-02-02 18:01:08,informatica_acessorios,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequente..."


É o mesmo comentário duas vezes. Ficamos com **uma linha por avaliação**, usando as informações do pedido mais recente, e guardamos todos os códigos de pedido na coluna `order_ids`. A coluna `multi_pedido` marca esses casos para que a escolha fique registrada.

In [10]:
codigos = avaliacoes.groupby("review_id")["order_id"].agg(lambda s: ",".join(sorted(set(s))))
avaliacoes = avaliacoes.sort_values("order_purchase_timestamp", ascending=False).drop_duplicates("review_id")
avaliacoes["order_ids"] = avaliacoes["review_id"].map(codigos)
avaliacoes["multi_pedido"] = avaliacoes["review_id"].map(repetidas) > 1

print(len(t["reviews"]), "linhas no arquivo original ->", len(avaliacoes), "avaliações únicas")
avaliacoes[avaliacoes.review_id == exemplo][["review_id", "order_ids", "multi_pedido", "categoria_pt"]]

99224 linhas no arquivo original -> 98410 avaliações únicas


,review_id,order_ids,multi_pedido,categoria_pt
29841,00130cbe1f9d422698c812ed8ded1919,"04a28263e085d399c97ae49e0b477efa,dfcdfc43867d1c1381bfaf62d6b9c195",True,informatica_acessorios


## 7. Passo 4: o texto das avaliações

Para cada avaliação montamos o texto em quatro etapas. Usamos as funções de `src/data/clean.py` e `src/data/pii.py`, que vamos testar primeiro com exemplos pequenos.

**a) Juntar título e comentário.** Quando os dois existem, viram uma frase só. Quando só um existe, usamos o que tiver.

In [11]:
print(juntar_titulo_mensagem("Ótimo", "Chegou antes do prazo"))
print(juntar_titulo_mensagem(None, "Chegou antes do prazo"))

Ótimo. Chegou antes do prazo
Chegou antes do prazo


**b) Substituir dados pessoais** (telefone, CPF, e-mail e CEP) por marcadores. Isso é feito logo no início, para que esses dados não cheguem a nenhum arquivo gerado.

In [12]:
mascarar_pii("Faltou um item, me ligue 34 99978 4951. Comprei em 2017 2018")

('Faltou um item, me ligue [TELEFONE]. Comprei em 2017 2018',
 {'[TELEFONE]': 1})

Repare que os anos não foram confundidos com telefone. A função devolve o texto e a contagem de substituições.

**c) Normalizar** o texto usado nas buscas: separar a pontuação, juntar pontuações repetidas e trocar algumas abreviações pela palavra inteira. Acentos e o "não" são mantidos, porque mudam o sentido da frase.

In [13]:
normalizar("n chegou!!! vc acredita? tbm veio sem nota")

'não chegou ! você acredita ? também veio sem nota'

**d) Marcar se o texto é útil.** Textos vazios, só com números ou pontuação, ou com menos de dois caracteres não entram nas buscas.

In [14]:
[(x, informativo(normalizar(x))) for x in ["10", "...", "A", "ok", "Produto ótimo"]]

[('10', False),
 ('...', False),
 ('A', False),
 ('ok', True),
 ('Produto ótimo', True)]

Aplicando as quatro etapas em todas as avaliações:

In [15]:
bruto = [juntar_titulo_mensagem(a, b) for a, b in zip(avaliacoes["review_comment_title"], avaliacoes["review_comment_message"])]
avaliacoes["texto_original"] = [mascarar_pii(x)[0] for x in bruto]
avaliacoes["texto_normalizado"] = avaliacoes["texto_original"].map(normalizar)
avaliacoes["n_palavras"] = avaliacoes["texto_normalizado"].map(contar_palavras)
avaliacoes["tem_titulo"] = avaliacoes["review_comment_title"].notna()
avaliacoes["tem_mensagem"] = avaliacoes["review_comment_message"].notna()
avaliacoes["tem_texto"] = avaliacoes["texto_original"] != ""
avaliacoes["indexavel"] = avaliacoes["texto_normalizado"].map(informativo)

avaliacoes[avaliacoes.tem_texto][["review_score", "texto_original", "texto_normalizado", "n_palavras", "indexavel"]].head()

,review_score,texto_original,texto_normalizado,n_palavras,indexavel
59696,1,"Troca. Por gentileza, trocar meu produto pois comprei Balde Spin Mop 360 Ces...","Troca . Por gentileza , trocar meu produto pois comprei Balde Spin Mop 360 C...",25,True
92235,5,Gostei. Os produtos são de boa qualidade e o preço que adquiri foi excelente.,Gostei . Os produtos são de boa qualidade e o preço que adquiri foi excelente .,14,True
92768,1,quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de v...,quero devolver o produto comprei um veio outro ; quero meu dinheiro de volta .,13,True
8486,1,o produto foi entregue errado eu estou esperado contatos da loja para resolv...,o produto foi entregue errado eu estou esperado contatos da loja para resolv...,15,True
45176,1,Razoavel. Peço um produto por um código e vem outro totalmente diferente.,Razoavel . Peço um produto por um código e vem outro totalmente diferente .,12,True


## 8. Passo 5: datas da avaliação

A tabela de avaliações tem duas datas. Na Olist, `review_creation_date` é o dia em que a pesquisa de satisfação foi **enviada** ao cliente, e `review_answer_timestamp` é quando ele **respondeu**. Guardamos as duas como números no formato aaaammdd, que facilitam filtros do tipo "entre tal e tal data".

In [16]:
def para_aaaammdd(serie):
    return pd.to_numeric(serie.dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")

avaliacoes["data_avaliacao"] = para_aaaammdd(avaliacoes["review_answer_timestamp"])
avaliacoes["data_envio_pesquisa"] = para_aaaammdd(avaliacoes["review_creation_date"])
avaliacoes[["review_creation_date", "data_envio_pesquisa", "review_answer_timestamp", "data_avaliacao"]].head(3)

,review_creation_date,data_envio_pesquisa,review_answer_timestamp,data_avaliacao
59696,2018-08-23,20180823,2018-08-24 02:56:26,20180824
92235,2018-06-27,20180627,2018-06-30 01:48:53,20180630
92768,2018-08-18,20180818,2018-08-19 00:39:54,20180819


## 9. Passo 6: os dois arquivos finais

Selecionamos as colunas finais e separamos a base em dois arquivos:

- **todas as avaliações** (`reviews.parquet`): usadas para contar e calcular médias, inclusive das avaliações sem comentário;
- **só as avaliações com texto útil** (`documents.parquet`): usadas nas buscas.

As duas têm exatamente as mesmas colunas. Guardar as avaliações sem comentário é importante porque mais da metade dos clientes só deu a nota, e perguntas sobre satisfação geral precisam considerar todos eles.

In [17]:
colunas = [
    "review_id", "order_ids", "review_score", "texto_original", "texto_normalizado",
    "tem_texto", "tem_titulo", "tem_mensagem", "indexavel", "n_palavras",
    "data_avaliacao", "data_envio_pesquisa", "ano_mes_compra", "order_status",
    "valor_total", "multi_pedido", "entregue", "dias_atraso", "entregue_com_atraso",
    "categoria_pt", "categoria_en", "multi_categoria", "n_itens", "uf_cliente",
]
base = avaliacoes[colunas].reset_index(drop=True)
documentos = base[base.indexavel].reset_index(drop=True)
print(f"todas as avaliações: {len(base):,} | com texto útil: {len(documentos):,}")

todas as avaliações: 98,410 | com texto útil: 42,121


### Dicionário das colunas

| Coluna | Significado |
|---|---|
| `review_id` | código da avaliação (é o que aparece como evidência nas respostas) |
| `order_ids` | código(s) do pedido avaliado, separados por vírgula |
| `review_score` | nota de 1 a 5 |
| `texto_original` | título e comentário como o cliente escreveu, com dados pessoais substituídos |
| `texto_normalizado` | versão do texto usada nas buscas |
| `tem_texto`, `tem_titulo`, `tem_mensagem` | se a avaliação tem algum texto, título ou comentário |
| `indexavel` | se o texto é útil para as buscas |
| `n_palavras` | número de palavras do comentário |
| `data_avaliacao` | data em que o cliente respondeu (aaaammdd) |
| `data_envio_pesquisa` | data em que a pesquisa foi enviada (aaaammdd) |
| `ano_mes_compra` | mês da compra (aaaamm) |
| `order_status` | situação do pedido (entregue, cancelado, etc.) |
| `valor_total` | soma de preço e frete dos itens do pedido, em reais |
| `multi_pedido` | se a mesma avaliação estava ligada a mais de um pedido |
| `entregue` | se o pedido foi entregue ao cliente |
| `dias_atraso` | dias entre a entrega real e a prevista (positivo = atraso; vazio se não entregue) |
| `entregue_com_atraso` | se a entrega passou da data prevista |
| `categoria_pt`, `categoria_en` | categoria do item de maior valor do pedido |
| `multi_categoria` | se o pedido tem produtos de mais de uma categoria |
| `n_itens` | quantidade de itens do pedido |
| `uf_cliente` | estado do cliente |

### Por que Parquet

Os arquivos são gravados em Parquet, e não em CSV, por três motivos. O formato guarda o tipo de cada coluna, então uma data ou um número não voltam como texto quando o arquivo é lido de novo. O arquivo fica menor, porque os dados são comprimidos coluna por coluna. E é possível ler só as colunas necessárias, sem carregar o arquivo inteiro.

## 10. Conferência com o script

Por fim, comparamos o resultado deste notebook com os arquivos gerados pelo `build_documents.py`. Se as etapas acima estão corretas, as duas bases devem ser idênticas.

In [18]:
gerado = pd.read_parquet(ROOT / "data" / "processed" / "reviews.parquet")

a = base.sort_values("review_id").reset_index(drop=True)
b = gerado.sort_values("review_id").reset_index(drop=True)
print("mesmo número de linhas:", len(a) == len(b))
print("mesmas colunas:", list(a.columns) == list(b.columns))
for col in ["review_score", "texto_original", "texto_normalizado", "indexavel", "categoria_pt",
            "valor_total", "dias_atraso", "entregue_com_atraso", "ano_mes_compra", "multi_pedido"]:
    iguais = a[col].astype(str).equals(b[col].astype(str))
    print(f"  {col:22s} {'igual' if iguais else 'DIFERENTE'}")

mesmo número de linhas: True
mesmas colunas: True
  review_score           igual
  texto_original         igual
  texto_normalizado      igual
  indexavel              igual
  categoria_pt           igual
  valor_total            igual
  dias_atraso            igual
  entregue_com_atraso    igual
  ano_mes_compra         igual
  multi_pedido           igual


Os resultados batem, então o script faz exatamente o que foi mostrado aqui. Para gerar a base novamente basta rodar, na raiz do projeto:

```bash
make data
```

O próximo notebook (`02_eda_reviews.ipynb`) analisa essa base em detalhe.